In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

In [2]:
import os

base_dir = r"D:\Breast_Cancer_Detection\Dataset"

In [3]:
# Data augmentation for training data
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)


In [4]:
# Validation data generator
val_datagen = ImageDataGenerator(rescale=1.0 / 255, validation_split=0.2)

In [5]:
# Create train and validation generators
train_generator = train_datagen.flow_from_directory(
    base_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = val_datagen.flow_from_directory(
    base_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 1263 images belonging to 3 classes.
Found 315 images belonging to 3 classes.


In [6]:
# Calculate class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights = dict(enumerate(class_weights))

print("Class Weights:", class_weights)


Class Weights: {0: np.float64(0.5904628330995793), 1: np.float64(1.2492581602373887), 2: np.float64(1.9765258215962442)}


In [7]:
# Build the CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')  # 3 classes: Benign, Malignant, Normal
])

d:\Breast_Cancer_Detection\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [9]:
# Train the model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    class_weight=class_weights
)


Epoch 1/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.4521 - loss: 1.0151 - val_accuracy: 0.5810 - val_loss: 0.8425
Epoch 2/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 19s 467ms/step - accuracy: 0.5281 - loss: 0.8231 - val_accuracy: 0.5810 - val_loss: 0.7327
Epoch 3/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 19s 463ms/step - accuracy: 0.4893 - loss: 0.7879 - val_accuracy: 0.6190 - val_loss: 0.7224
Epoch 4/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 19s 462ms/step - accuracy: 0.5748 - loss: 0.7374 - val_accuracy: 0.6381 - val_loss: 0.6996
Epoch 5/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 19s 468ms/step - accuracy: 0.5819 - loss: 0.7298 - val_accuracy: 0.6317 - val_loss: 0.7235
Epoch 6/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 19s 472ms/step - accuracy: 0.5740 - loss: 0.7300 - val_accuracy: 0.6476 - val_loss: 0.7186
Epoch 7/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 18s 452ms/step - accuracy: 0.6342 - loss: 0.7046 - val_accuracy: 0.6032 - val_loss: 0.8002
Epoch 8/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 19s 471ms/step - accuracy: 0.6239 - loss: 0.6834 - val_accurac

In [17]:
import os

os.makedirs("model", exist_ok=True)
model.save(os.path.join("model", "breast_cancer_model.h5"))

In [11]:
# Evaluate the model
loss, accuracy = model.evaluate(val_generator)
print(f"Validation Loss: {loss}")
print(f"Validation Accuracy: {accuracy}")


10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 193ms/step - accuracy: 0.7905 - loss: 0.5729
Validation Loss: 0.5729206800460815
Validation Accuracy: 0.7904762029647827


In [17]:
import os

os.makedirs("model", exist_ok=True)

model.save("model/breast_cancer_model.h5")

print("Model saved successfully!")

Model saved successfully!


In [19]:
# Plot training history
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()


ModuleNotFoundError: No module named 'matplotlib'